# Dependencies

In [198]:
from datasets import load_dataset
from dotenv import load_dotenv
from typing import Callable, Any
from pathlib import Path
import torch as T
import string
import os

# Data
Importing the data

In [199]:
load_dotenv("./.env.secrets")

hf_token = os.getenv("HF_TOKEN")

FORMAT = "torch"

ds = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-v1",
    token=hf_token if hf_token else False,
).with_format(FORMAT)

x_test, x_train, x_val = (ds[key] for key in ds.keys())

display(x_train["text"][:10])

['',
 ' = Valkyria Chronicles III = \n',
 '',
 ' Senjō no Valkyria 3 : <unk> Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " <unk> Raven " . \n',
 " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving

## Data pre-processing
Preprocessing the data allows us to collect as much useful information as possible, these following operations are performed:

- Remove punctuations: Since our aim is to analyze the semantic meaning of individual words and not sentences. Including punctuation, which carries meaning in a sentences, would be misleading since we would lead to situations where two different words (for example "apple." and "apple") have slightly different meanings when they, when taken out of a sentence context, should have practically the same meaning. This is also computationally inficient if you consider that our task is strictly confined to analyzing the semantic difference and meaning of words.

- Force lower: We do this for exactly the same reasons as why we remove punctuation.

You could make the case that the words "Apple", "apple", "apple." actually do have differenet semanting meanings and technically you would be correct, but as stated above, we want to focus the model to learn the difference between "apple" and "pear" rather than using its parameters to differentiate between "Apple" and "apple".

In [200]:
def pre_process_data(data: T.utils.data.Dataset):
    # We clean the words in the dataset a bit, this is to ensure that all words present
    # in the vocabulary are in a predictable format
    to_remove = string.punctuation + "“”‘’"
    table = str.maketrans("", "", to_remove)  # Create a translation table
    big_blob = " ".join(data["text"])
    # Translate the the blob using the table and convert to lower
    cleaned_blob = big_blob.translate(table).lower()
    return sorted(list(set(cleaned_blob.split())))

# One-hot-encoding model
Assigns each word in a vocabulary a $n$ dimensional vector with a single element set to one (uniquely), the words are sorted alphabetically.

In [201]:
class OneHotEncoder:
    def __init__(self, vocabulary: list[str] | OneHotEncoder):
        if type(vocabulary) is OneHotEncoder:
            self.dim = vocabulary.dim
            self.word_indexes = vocabulary.word_indexes
        else:
            self.dim = len(vocabulary)
            self.word_indexes = {word: i for i, word in enumerate(vocabulary)}

    def __call__(self, word: str):
        vec = T.zeros(self.dim)
        idx = self.word_indexes.get(word)
        if idx:
            vec[idx] = 1
        return vec

In [202]:
FN = "one_hot_encoder.pt"
file_path = Path(FN)
if file_path.exists():
    encoder = T.load(file_path, weights_only=False)
else:
    vocabulary = pre_process_data(x_train)
    encoder = OneHotEncoder(vocabulary)
    T.save(encoder, file_path)

# Creating the model
In a Object Oriented way

## Softmax object

In [203]:
class Softmax(Callable[[T.Tensor], T.Tensor]):
    """
    Row based Softmax implementation, this means it summarizes
    across the first dimension.
    """
    def __call__(self, z: T.Tensor) -> T.Tensor:
        z_exp = T.exp(z) # Exponentiate all elements of z
        return z_exp / T.sum(z_exp, dim=0)


sm = Softmax()
display(
    sm(T.tensor([1, 0, 0, 0]).reshape(-1, 1)),
    sm(T.tensor([1, 1, 0, 0]).reshape(-1, 1)),
    sm(T.tensor([1, 1, 1, 0]).reshape(-1, 1)),
    sm(T.tensor([1, 1, 1, 1]).reshape(-1, 1)),
)

tensor([[0.4754],
        [0.1749],
        [0.1749],
        [0.1749]])

tensor([[0.3655],
        [0.3655],
        [0.1345],
        [0.1345]])

tensor([[0.2969],
        [0.2969],
        [0.2969],
        [0.1092]])

tensor([[0.2500],
        [0.2500],
        [0.2500],
        [0.2500]])

## LinearLayer object

In [229]:
from numpy import dtype


class LinearLayerD1:
    def __init__(
        self,
        shape: list[int],
        activation: Callable[[T.Tensor], T.Tensor],
    ):
        self.shape = shape
        # Add bias dim
        weight_dims = shape
        weight_dims[0] += 1
        self.weights = T.randn(
            size=weight_dims,
        )
        self.activation = activation

    def __call__(self, vec: T.Tensor) -> T.Tensor:
        # add 1 to the last dim and fill that with 1s to create a single
        # weight matrix
        ones = T.ones(1,)
        vec_plus_bias = T.cat((vec, ones), dim=0)
        print(vec_plus_bias)
        return self.activation((vec_plus_bias.T @ self.weights))


vec = T.tensor([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1], dtype=T.float32)

display(
    LinearLayerD1([10, 3], Softmax())(vec)
)

tensor([0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000, 1.0000])


tensor([0.0885, 0.4065, 0.5049])

## Model

In [ ]:
class Model():
    def __init__(self, layers: list):
        self.layers = layers

    def __call__(self, input: Any):
        curr = input
        for l in self.layers:
            curr = l(curr)
        return curr

# Word2Vec model

In [ ]:
ONE_HOT_DIM = encoder.dim
OUT_DIM = 10
word2vec = Model([
    OneHotEncoder(encoder), 
    LinearLayerD1(
        shape=(ONE_HOT_DIM,OUT_DIM),
        activation=Softmax()
    )
])

display(word2vec("banana"))

tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])